# Phase 1 analysis: retained results and quality evidence

This notebook collects the existing raw-temperature analysis and retained Phase 1 quality findings. It reads current and historical inputs only; no transformation, audit, publication, receipt or report is generated.

The raw plots use the retained temperature aggregate, whereas the source-only production and duplicate investigation explicitly exclude that aggregate. The two populations are not interchangeable. Historical files are checked against their recorded SHA-256 identities; current production files are checked against producer status. Missing or changed inputs fail rather than triggering a fallback or rebuild.


## Input identity and provenance

Historical hashes identify the retained files, not a claim that the original exporter or source lineage is fully known. In particular, the exporter for `key_statistics.csv` was not identified in the baseline inventory. Production status is a producer record, not an independent receipt.

In [ ]:
from __future__ import annotations

from pathlib import Path
import hashlib
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'config' / 'transformation_defaults.yaml').is_file():
    if REPO_ROOT.parent == REPO_ROOT:
        raise FileNotFoundError('Run from the repository root or scripts directory.')
    REPO_ROOT = REPO_ROOT.parent
DATA_DIR = REPO_ROOT / 'Data' / 'files_csv'
TEMPERATURE_PATH = DATA_DIR / 'temperature_raw.csv'
OUTPUTS = REPO_ROOT / 'artifacts' / 'phase1_outputs'
DUPLICATE = OUTPUTS / 'duplicate'
HISTORICAL_INPUTS = {
    TEMPERATURE_PATH: 'b15f0ca9e51fc5b79849c39f3c54192217d960065e286e5c638edd17e2b4f4e7',
    OUTPUTS / 'key_statistics.csv': 'a7169c440b69c4a0581518aa92d4f65a77b66fc9d00985bd1230aa9d3fe0ea06',
    DUPLICATE / 'phase1_source_exclusion_comparison.json': 'fb4258c99998384c9d722e296f807c9e5605d178d6983721fd98497ea0240c48',
    DUPLICATE / 'qc_phase1_duplicate_summary.json': 'b96c3c2afedc3e47615a5b9d349da3de2a474e7be990d238ed1136bdab4e0f5f',
    DUPLICATE / 'qc_phase1_duplicate_conflicts.csv': '137e26f2b9321d2dc9a6d59431ef073989d4c884f74c995cd4cf08170b3ed45d',
    DUPLICATE / 'excluding_temperature_raw' / 'qc_phase1_duplicate_summary.json': '7bb69481afd365ae936305564ad951409a3796d24305e44d1de85431eb7c0310',
    DUPLICATE / 'excluding_temperature_raw' / 'qc_phase1_duplicate_conflicts.csv': '1bfe07fc0d9ba2fdfa9869e6f011e58514028b3966adb24d291b7b0f89921c6d',
    DUPLICATE / 'excluding_temperature_raw' / 'qc_phase1_unmatched_records.csv': '3967db40d752a26cf5af306eb6b73c29e29ce4d41854d77c8c4a36a1c9b3984f',
}

def sha256_file(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

for path, expected in HISTORICAL_INPUTS.items():
    if not path.is_file():
        raise FileNotFoundError(f'Required retained historical input is missing: {path}')
    if sha256_file(path) != expected:
        raise ValueError(f'Retained historical input has changed; verify its lineage before analysis: {path}')

publication_path = REPO_ROOT / 'artifacts' / 'phase1_operational' / 'phase1_run.json'
publication = json.loads(publication_path.read_text(encoding='utf-8'))
if (publication.get('schema_version') != 1 or publication.get('status') != 'passed'
        or publication.get('limited_run') is not False or publication.get('quality_gates_passed') is not True):
    raise ValueError('Phase 1 production status is missing, failed, incomplete, or limited.')
for path, expected in (
    (REPO_ROOT / 'scripts' / 'phase1_implement.ipynb', publication.get('implementation_sha256')),
    (REPO_ROOT / 'config' / 'transformation_defaults.yaml', publication.get('config_sha256')),
    (REPO_ROOT / 'artifacts' / 'phase1_operational' / 'phase1_aligned_temperature.csv', publication.get('hashes', {}).get('phase1_aligned_sha256')),
    (REPO_ROOT / 'artifacts' / 'phase1_operational' / 'phase1_source_inventory.csv', publication.get('hashes', {}).get('raw_index_sha256')),
):
    if not path.is_file() or sha256_file(path) != expected:
        raise ValueError(f'Phase 1 publication is stale or its required file is missing: {path}')

files = [TEMPERATURE_PATH]
print(f'Using retained temperature-only aggregate: {TEMPERATURE_PATH}')
print(f'Repository root: {REPO_ROOT}')
files[:3]


## Load the retained temperature aggregate

The original baseline ran with this aggregate already present. No source-file fallback or aggregate creation is performed.

In [ ]:
frames = []

for f in files:
    df = pd.read_csv(f)
    required = {'Time', 'DeviceId', 'Sensor', 'Value'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'{f.name} missing columns: {sorted(missing)}')

    df = df.copy()
    df['source_file'] = f.name
    df['Time'] = pd.to_numeric(df['Time'], errors='coerce')
    df['Value'] = pd.to_numeric(df['Value'], errors='coerce')
    df['DeviceId'] = df['DeviceId'].astype(str)
    df['Sensor'] = df['Sensor'].astype(str)
    df['ts_utc'] = pd.to_datetime(df['Time'], unit='s', utc=True)
    df['minute_bucket'] = df['ts_utc'].dt.floor('min')
    frames.append(df)

raw = pd.concat(frames, ignore_index=True, sort=False)
raw = raw.sort_values(['ts_utc', 'DeviceId', 'Sensor']).reset_index(drop=True)
print(f'Loaded {len(raw):,} temperature rows from {TEMPERATURE_PATH}')
temperature_raw = raw.copy()
raw.head()


## Aggregate scope and unique timestamps

This count and all charts through the credibility table use only the retained `temperature_raw.csv` aggregate, including its original duplicate observations. It is not the source-only production population.

In [ ]:
unique_time_count = temperature_raw['Time'].nunique()
print(f'Unique Time values in raw: {unique_time_count}')

## Reporting cadence

Positive differences between consecutive timestamps are grouped by device and sensor; zero gaps are excluded, as in the original analysis.

In [ ]:
gap_seconds = (
    raw.sort_values(['DeviceId', 'Sensor', 'Time'])
       .groupby(['DeviceId', 'Sensor'])['Time']
       .diff()
       .dropna()
       .astype(float)
)

gap_seconds = gap_seconds[gap_seconds > 0]

print('Observed inter-arrival time distribution (seconds):')
display(gap_seconds.value_counts().sort_index().head(15))

plt.figure(figsize=(10, 5))
gap_seconds.plot(kind='hist', bins=range(0, 15), edgecolor='black')
plt.title('Inter-arrival time histogram by device and sensor')
plt.xlabel('Seconds between consecutive readings')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## First five minutes: reports by device

The existing chart shows unique device/second reports and ten-second guides in the first five-minute interval.

In [ ]:
first_five_minutes_start = temperature_raw['ts_utc'].min()
first_five_minutes_end = first_five_minutes_start + pd.Timedelta(minutes=5)
first_five_minutes = temperature_raw.loc[
    temperature_raw['ts_utc'].between(
        first_five_minutes_start,
        first_five_minutes_end,
        inclusive='left',
    ),
    ['Time', 'ts_utc', 'DeviceId'],
].drop_duplicates(['Time', 'DeviceId'])

device_order = sorted(first_five_minutes['DeviceId'].unique())
row_spacing = 0.55
device_positions = {
    device: position * row_spacing
    for position, device in enumerate(device_order)
}
y_positions = [device_positions[device] for device in device_order]

second_marks = pd.date_range(
    first_five_minutes_start,
    first_five_minutes_end,
    freq='10s',
    inclusive='left',
)

plt.figure(figsize=(14, 4.5))
plt.vlines(
    second_marks,
    ymin=-0.4,
    ymax=y_positions[-1] + 0.4,
    colors='#5b6573',
    linewidth=0.7,
    alpha=0.85,
    zorder=0,
)
plt.scatter(
    first_five_minutes['ts_utc'],
    first_five_minutes['DeviceId'].map(device_positions),
    marker='s',
    s=100,
    alpha=1,
    linewidths=0,
    zorder=1,
)
plt.yticks(y_positions, device_order)
plt.ylim(-0.4, y_positions[-1] + 0.4)
plt.xlabel('Time (UTC)')
plt.ylabel('DeviceId')
plt.title('Temperature reporting by device during the first five minutes')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print(f'First five minutes: {first_five_minutes_start} to {first_five_minutes_end}')
print(f'Guide lines drawn: {len(second_marks)} at 10-second intervals')
print(f'Unique device-second reports plotted: {len(first_five_minutes):,}')

## First five minutes: 30-second completeness

A filled block means at least one report was observed for that device in the 30-second interval; it does not count how many reports occurred.

In [ ]:
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

block_size = pd.Timedelta(seconds=30)
block_edges = pd.date_range(
    first_five_minutes_start,
    first_five_minutes_end,
    freq=block_size,
)
block_count = len(block_edges) - 1

reports_30s = first_five_minutes.copy()
reports_30s['block_index'] = (
    (reports_30s['ts_utc'] - first_five_minutes_start).dt.total_seconds() // 30
).astype(int)

report_matrix = (
    reports_30s.assign(reported=1)
    .pivot_table(
        index='DeviceId',
        columns='block_index',
        values='reported',
        aggfunc='max',
        fill_value=0,
    )
    .reindex(index=device_order, columns=range(block_count), fill_value=0)
)

block_y_centers = [device_positions[device] for device in device_order]
block_y_edges = [
    block_y_centers[0] - row_spacing / 2,
    *[(left + right) / 2 for left, right in zip(block_y_centers, block_y_centers[1:])],
    block_y_centers[-1] + row_spacing / 2,
]

fig, ax = plt.subplots(figsize=(14, 4.5))
ax.pcolormesh(
    block_edges,
    block_y_edges,
    report_matrix.to_numpy(),
    cmap=ListedColormap(['#eef1f4', '#2878b5']),
    vmin=0,
    vmax=1,
    edgecolors='#5b6573',
    linewidth=0.8,
    shading='flat',
)
ax.set_yticks(block_y_centers, device_order)
ax.set_ylim(block_y_edges[0], block_y_edges[-1])
ax.set_xlabel('30-second reporting interval (UTC)')
ax.set_ylabel('DeviceId')
ax.set_title('Temperature reporting by device in 30-second blocks')
ax.legend(
    handles=[
        Patch(facecolor='#2878b5', edgecolor='#5b6573', label='Reported at least once'),
        Patch(facecolor='#eef1f4', edgecolor='#5b6573', label='No report observed'),
    ],
    loc='upper right',
)
fig.tight_layout()
plt.show()

print(f'30-second blocks: {block_count}')
print(f'Blocks with at least one report: {int(report_matrix.to_numpy().sum()):,}')

## Temperature sample counts

Counts include the retained aggregate's duplicate observations.

In [ ]:
samples_by_device = (
    temperature_raw.groupby('DeviceId')
    .size()
    .sort_values(ascending=False)
)

ax = samples_by_device.plot(
    kind='bar',
    figsize=(10, 5),
    color='#2878b5',
    edgecolor='#1f2933',
    linewidth=0.8,
)
ax.set_title('Temperature samples reported by device')
ax.set_xlabel('DeviceId')
ax.set_ylabel('Number of samples')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

print(f'Total temperature samples counted: {int(samples_by_device.sum()):,}')

## Temperature distributions by device

The original boxplots omit displayed outliers; they do not remove observations from the underlying calculations.

In [ ]:
device_temperature = [
    temperature_raw.loc[temperature_raw['DeviceId'].eq(device), 'Value'].dropna()
    for device in device_order
]

fig, axes = plt.subplots(2, 4, figsize=(16, 8), sharey=True)
axes = axes.ravel()

for ax, device, values in zip(axes, device_order, device_temperature):
    ax.boxplot(
        values,
        patch_artist=True,
        showfliers=False,
        boxprops={'facecolor': '#2878b5', 'alpha': 0.75},
        medianprops={'color': '#d62728', 'linewidth': 2},
    )
    ax.set_title(f'Device {device}')
    ax.set_xticks([])
    ax.set_ylabel('Temperature' if ax in axes[[0, 4]] else '')

fig.suptitle("Temperature Distribution by Device", fontsize=16)
fig.supxlabel("DeviceId")
fig.supylabel("Temperature")
fig.tight_layout()
plt.show()

## Temperature over time

The existing time series plots all retained aggregate temperature samples by device.

In [ ]:
all_devices = sorted(temperature_raw['DeviceId'].unique())

fig, ax = plt.subplots(figsize=(16, 7))
for device in all_devices:
    device_data = temperature_raw.loc[
        temperature_raw['DeviceId'].eq(device)
    ].sort_values('ts_utc')
    ax.plot(
        device_data['ts_utc'],
        device_data['Value'],
        linewidth=0.6,
        alpha=0.75,
        label=f'Device {device}',
    )

ax.set_title('Temperature over time by device')
ax.set_xlabel('Time (UTC)')
ax.set_ylabel('Temperature')
ax.legend(title='DeviceId')
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

print(f'Device series plotted: {len(all_devices)}')
print(f'Total temperature samples plotted: {len(temperature_raw):,}')

## Per-device descriptive statistics

Mean, median, sample standard deviation and quartiles are the original aggregate-based statistics, rounded to two decimals.

In [ ]:
device_temperature_summary = (
    temperature_raw.groupby('DeviceId')['Value']
    .agg(
        mean='mean',
        median='median',
        standard_deviation='std',
        first_quarter=lambda values: values.quantile(0.25),
        third_quarter=lambda values: values.quantile(0.75),
    )
    .sort_index()
    .round(2)
)

device_temperature_summary

## Dataset overview

`raw_files` describes the aggregate file loaded by this notebook, not the number of source files used in production.

In [ ]:
n_rows = len(raw)
n_files = raw['source_file'].nunique()
n_devices = raw['DeviceId'].nunique()
n_sensors = raw['Sensor'].nunique()
start_ts = raw['ts_utc'].min()
end_ts = raw['ts_utc'].max()
span_hours = (end_ts - start_ts).total_seconds() / 3600

overview = pd.DataFrame({
    'metric': [
        'raw_rows',
        'raw_files',
        'unique_devices',
        'unique_sensors',
        'time_start_utc',
        'time_end_utc',
        'span_hours'
    ],
    'value': [
        n_rows,
        n_files,
        n_devices,
        n_sensors,
        start_ts,
        end_ts,
        round(span_hours, 2)
    ]
})
overview

## Records per minute

The original summary describes the number of aggregate rows in each observed minute; it is not a completeness calculation against absent minutes.

In [ ]:
minute_counts = raw.groupby('minute_bucket').size().rename('records_per_minute')
minute_counts.describe()

## Existing variability/credibility score

This reproduces the notebook's existing coefficient-of-variation-based score. It is a descriptive heuristic, not an independent validation of sensor accuracy.

In [ ]:
def score_monitoring_credibility(df: pd.DataFrame) -> pd.DataFrame:
    variability = (
        df.groupby('Sensor')['Value']
          .agg(['mean', 'std'])
          .reset_index()
    )
    variability['cv'] = variability['std'] / variability['mean'].replace(0, np.nan)
    variability['variability_score'] = np.clip(100 - (variability['cv'].fillna(0) * 100), 0, 100)
    variability['credibility_score'] = variability['variability_score']
    return variability.sort_values('credibility_score', ascending=False)

credibility = score_monitoring_credibility(raw)
credibility

## Current source-only production QC

These gate results describe the current general Phase 1 transformation, which excludes `temperature_raw.csv`. The average statistics apply the original strict confidence filter and all observed devices. They are not the monitoring-cohort P90.


In [ ]:
display(pd.DataFrame(publication['gate_checks']))
display(pd.DataFrame([publication['qc']]))
display(pd.DataFrame(publication['general_statistics']))


## Retained historical per-device statistics

This retained user-facing table is presented as saved. Its exporter was not identified in the recovery baseline; it is not silently treated as a newly produced artifact.


In [ ]:
display(pd.read_csv(OUTPUTS / 'key_statistics.csv'))

## Duplicate investigations and reconciliation

The **aggregate-inclusive** historical audit includes both the aggregate and individual source files. The **source-only** audit excludes the aggregate from duplicate statistics and separately reconciles it against timestamped sources. Their duplicate rates therefore use different populations. The detailed conflict examples and exhaustive unmatched export remain linked below; the large unmatched CSV is not loaded into a dataframe.

- [Aggregate-inclusive conflict examples](../artifacts/phase1_outputs/duplicate/qc_phase1_duplicate_conflicts.csv)
- [Source-only conflict examples](../artifacts/phase1_outputs/duplicate/excluding_temperature_raw/qc_phase1_duplicate_conflicts.csv)
- [Exhaustive unmatched records](../artifacts/phase1_outputs/duplicate/excluding_temperature_raw/qc_phase1_unmatched_records.csv)
- [Source-exclusion comparison](../artifacts/phase1_outputs/duplicate/phase1_source_exclusion_comparison.json)


In [ ]:
inclusive = json.loads((DUPLICATE / 'qc_phase1_duplicate_summary.json').read_text(encoding='utf-8'))
source_only = json.loads((DUPLICATE / 'excluding_temperature_raw' / 'qc_phase1_duplicate_summary.json').read_text(encoding='utf-8'))
comparison = json.loads((DUPLICATE / 'phase1_source_exclusion_comparison.json').read_text(encoding='utf-8'))
fields = ['rows_temperature', 'unique_keys', 'duplicate_rows', 'duplicate_key_count',
          'duplicate_key_rate_percent', 'identical_value_duplicate_keys',
          'conflicting_value_duplicate_keys', 'cross_file_duplicate_keys']
display(pd.DataFrame({name: {key: report[key] for key in fields} for name, report in (
    ('aggregate-inclusive historical audit', inclusive),
    ('source-only historical audit', source_only),
)}))
display(pd.json_normalize(source_only['aggregate_reconciliation']).T.rename(columns={0: 'value'}))
display(pd.DataFrame([{
    key: comparison[key] for key in ('before_sha256', 'after_sha256', 'before_rows', 'after_rows',
                                     'columns_unchanged', 'timestamp_order_unchanged',
                                     'sensor_temperature_cells_changed', 'availability_cells_changed')
}]))
